# Sales Data Analysis Test Notebook

This notebook demonstrates a simple data science workflow for testing the context retrieval persona.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [3]:
# Create sample sales data
np.random.seed(42)
n_samples = 1000

data = {
    'advertising_spend': np.random.uniform(1000, 50000, n_samples),
    'sales_team_size': np.random.randint(5, 50, n_samples),
    'market_size': np.random.uniform(100000, 1000000, n_samples),
    'season': np.random.choice(['Q1', 'Q2', 'Q3', 'Q4'], n_samples)
}

# Generate revenue with some realistic relationships
data['revenue'] = (
    data['advertising_spend'] * 2.5 + 
    data['sales_team_size'] * 1000 + 
    data['market_size'] * 0.1 +
    np.random.normal(0, 10000, n_samples)
)

df = pd.DataFrame(data)
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (1000, 5)


,advertising_spend,sales_team_size,market_size,season,revenue
0,19352.465824,16,788328.820813,Q3,136533.340119
1,47585.001014,20,257354.764534,Q2,174753.331248
2,36867.703149,28,552309.468817,Q1,163944.822574
3,30334.265726,23,458796.724995,Q4,131081.610566
4,8644.913382,12,231736.592946,Q2,63862.306986


In [4]:
# Prepare data for modeling
# One-hot encode categorical variables
df_encoded = pd.get_dummies(df, columns=['season'], prefix='season')

# Define features and target
feature_columns = [col for col in df_encoded.columns if col != 'revenue']
X = df_encoded[feature_columns]
y = df_encoded['revenue']

print(f"Features: {X.columns.tolist()}")
print(f"Target: revenue")
print(f"Feature matrix shape: {X.shape}")

Features: ['advertising_spend', 'sales_team_size', 'market_size', 'season_Q1', 'season_Q2', 'season_Q3', 'season_Q4']
Target: revenue
Feature matrix shape: (1000, 7)


In [5]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

Training set size: 800
Test set size: 200


In [6]:
# AutoGluon Tabular ML Solution - Dataset Specific
from autogluon.tabular import TabularDataset, TabularPredictor

# Dataset Analysis:
# - Shape: (1 columns detected)
# - Target Column: 'numeric_cols'
# - Available Columns: ['season']
# - Problem Type: None

# Verify target column exists
if 'numeric_cols' not in X_train.columns:
    print("⚠️  Target column 'numeric_cols' not found!")
    print("Available columns:", list(X_train.columns))
    # Try to find a suitable target column
    numeric_cols = X_train.select_dtypes(include=['number']).columns.tolist()
    if len(numeric_cols) > 0:
        actual_target = numeric_cols[-1]  # Use last numeric column
        print(f"Using '{actual_target}' as target instead")
    else:
        actual_target = X_train.columns[-1]  # Use last column
        print(f"Using '{actual_target}' as target instead")
else:
    actual_target = 'numeric_cols'

print(f"📊 Training with target column: {actual_target}")
print(f"📋 Dataset shape: {X_train.shape}")

# Load your data
train_data = TabularDataset(X_train)

# Train AutoGluon model
predictor = TabularPredictor(
    label=actual_target,
    # problem_type=None (auto-detect),
    path='./autogluon_models/tabular_model'
).fit(
    train_data,
    time_limit=120,
    presets='best_quality'
)

print(f"✅ Training completed for {actual_target}!")

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.11
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 24.5.0: Tue Apr 22 19:54:29 PDT 2025; root:xnu-11417.121.6~2/RELEASE_ARM64_T6030
CPU Count:          12
Memory Avail:       7.03 GB / 36.00 GB (19.5%)
Disk Space Avail:   297.84 GB / 460.43 GB (64.7%)
Presets specified: ['best_quality']
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the optimal `num_stack_levels` value. Copies of AutoGluon will be fit on subsets of the data. 

⚠️  Target column 'numeric_cols' not found!
Available columns: ['advertising_spend', 'sales_team_size', 'market_size', 'season_Q1', 'season_Q2', 'season_Q3', 'season_Q4']
Using 'market_size' as target instead
📊 Training with target column: market_size
📋 Dataset shape: (800, 7)


	Running DyStack sub-fit in a ray process to avoid memory leakage. Enabling ray logging (enable_ray_logging=True). Specify `ds_args={'enable_ray_logging': False}` if you experience logging issues.
2025-07-29 13:06:49,813	INFO worker.py:1843 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 
		Context path: "/Users/jujonahj/jupyter-ai-personas/jupyter_ai_personas/data_science_persona/autogluon_models/tabular_model/ds_sub_fit/sub_fit_ho"
(_dystack pid=80532) Running DyStack sub-fit ...
(_dystack pid=80532) Beginning AutoGluon training ... Time limit = 28s
(_dystack pid=80532) AutoGluon will save models to "/Users/jujonahj/jupyter-ai-personas/jupyter_ai_personas/data_science_persona/autogluon_models/tabular_model/ds_sub_fit/sub_fit_ho"
(_dystack pid=80532) Train Data Rows:    711
(_dystack pid=80532) Train Data Columns: 6
(_dystack pid=80532) Label Column:       market_size
(_dystack pid=80532) Problem Type:       regression
(_dystack pid=80532) Preprocessing da

✅ Training completed for market_size!


In [7]:
# 🏆 VIEW MODEL LEADERBOARD AND BEST MODELS
leaderboard = predictor.leaderboard()
print("🏆 AutoGluon Model Leaderboard:")
print("="*50)
print(leaderboard.head(10))  # Show top 10 models

# 🥇 BEST MODEL INFORMATION
best_model = leaderboard.iloc[0]['model']
best_score = leaderboard.iloc[0]['score_val']
print(f"\n🥇 BEST MODEL: {best_model}")
print(f"📊 BEST SCORE: {best_score:.4f}")

# 📈 DETAILED RANKING
print("\n📈 Top 5 Models Ranking:")
for i, row in leaderboard.head(5).iterrows():
    print(f"{i+1:2d}. {row['model']:25s} | Score: {row['score_val']:.4f} | Time: {row['fit_time']:.1f}s")

🏆 AutoGluon Model Leaderboard:
                         model      score_val              eval_metric  \
0          WeightedEnsemble_L2 -253279.875826  root_mean_squared_error   
1  NeuralNetFastAI_r145_BAG_L1 -254528.470790  root_mean_squared_error   
2  NeuralNetFastAI_r102_BAG_L1 -255104.129071  root_mean_squared_error   
3       NeuralNetFastAI_BAG_L1 -255211.960090  root_mean_squared_error   
4    NeuralNetTorch_r22_BAG_L1 -256077.139599  root_mean_squared_error   
5  NeuralNetFastAI_r191_BAG_L1 -256568.441630  root_mean_squared_error   
6    NeuralNetTorch_r79_BAG_L1 -256951.519113  root_mean_squared_error   
7        NeuralNetTorch_BAG_L1 -257134.467329  root_mean_squared_error   
8          CatBoost_r13_BAG_L1 -258390.973521  root_mean_squared_error   
9           CatBoost_r9_BAG_L1 -258419.483539  root_mean_squared_error   

   pred_time_val   fit_time  pred_time_val_marginal  fit_time_marginal  \
0       0.086512  10.366076                0.000180           0.004326   
1     

In [ ]:
# Train a simple linear regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Make predictions
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# Calculate metrics
train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

print("Model Performance:")
print(f"Training MSE: {train_mse:,.2f}")
print(f"Test MSE: {test_mse:,.2f}")
print(f"Training R²: {train_r2:.4f}")
print(f"Test R²: {test_r2:.4f}")